# specs_novel — G0.1-R: re-embed the corpus with the pinned DreaMS (Colab, GPU)

**Generated by `build/mk_novel_p0.py` — do not hand-edit.** Pre-registered in
`specs_novel/gate_report.md` → *G0.1-R*.

G0 came back HOLD on one leg: the cached `_dreams_out/emb/` vectors are ≥ 0.998 but not ≥ 0.999
from the pinned DreaMS on the parity rows, on Colab CPU and Kaggle GPU alike, so the cache was built
by other code (decision rule 1). This notebook rebuilds the whole corpus embedding with the
vendored `dreams_infer` + `dreams_embed.pt` — the exact code path the Kaggle notebook verified — so
training embeddings and test-time query embeddings come from one implementation.

- **Rows and order:** the P0.2 replay (`_novel/p0/corpus_ce.parquet` → `train_row`), which matched
  `corpus_meta.parquet` row by row. Row `i` of the new cache is row `i` of `corpus_meta.parquet`.
- **Output:** `casmi_2026/_novel/emb_dbec3a0/emb_*.npy` (200,000 rows per chunk, float32, 1024-d)
  + `manifest.json`. The old cache is only read, never written.
- **Resumable:** a chunk already on Drive with the right shape is skipped, so a disconnected
  runtime just reruns all cells.
- **Checks:** G0.1-R (every parity row ≥ 0.999 vs Colab (b), rebuilt parity peaks identical), and —
  recorded, not gating — the new-vs-old cosine over the whole corpus.

Runtime: **GPU with High-RAM** (T4 ≈ 2.5 h of embedding; L4 / A100 faster) + ~15 min of peak prep.

In [ ]:
import os, sys, json, time, shutil, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/casmi_2026")
TRAIN = ROOT / "enveda-CASMI26-molecule-id-mass-spectra" / "train.parquet"
DOUT = ROOT / "_dreams_out"                     # the old cache: corpus_meta.parquet, emb/ (read only)
GATE = ROOT / "_novel" / "p0"                   # P0 outputs: corpus_ce.parquet, dreams_embed.pt, parity_200.npz
OUT = ROOT / "_novel" / "emb_dbec3a0"           # the new cache
WORK = Path("/content/novel_reembed")
SRC = WORK / "src"
for p in (OUT, WORK, SRC):
    p.mkdir(parents=True, exist_ok=True)

DREAMS_COMMIT = "dbec3a0b514a99e5056cfccde4559fda8cfe8129"
CHUNK = 200_000                                 # rows per emb_*.npy, as in the old cache
DIM = 1024
BATCH = 256
COS_CACHE = 0.999                               # G0.1-R: new cache vs Colab (b), every parity row
KAGGLE_COS_MIN = 0.999999999995092              # G0.1 Kaggle leg, from phase0_kaggle.json (2026-09-29)

P0 = json.loads((GATE / "phase0_colab_gate.json").read_text())
MAN = P0["upload"]["manifest"]
assert MAN["dreams_commit"] == DREAMS_COMMIT, MAN["dreams_commit"]
RESULT = {"started": time.strftime("%Y-%m-%d %H:%M"), "dreams_commit": DREAMS_COMMIT,
          "dreams_embed_sha1": MAN["dreams_embed_sha1"], "out": str(OUT), "batch_size": BATCH}

def save_result():
    (GATE / "reembed_gate.json").write_text(json.dumps(RESULT, indent=1, default=str))

assert torch.cuda.is_available(), "switch the runtime to a GPU: the corpus is ~2.5 GPU-hours on a T4"
torch.backends.cuda.matmul.allow_tf32 = False   # plain float32, as on the Kaggle leg
torch.backends.cudnn.allow_tf32 = False
DEV = "cuda:0"
RESULT["gpu"] = torch.cuda.get_device_name(0)
RESULT["torch"] = torch.__version__
for name, p in [("train.parquet", TRAIN), ("corpus_meta.parquet", DOUT / "corpus_meta.parquet"),
                ("old emb/", DOUT / "emb"), ("corpus_ce.parquet", GATE / "corpus_ce.parquet"),
                ("dreams_embed.pt", GATE / "dreams_embed.pt"), ("parity_200.npz", GATE / "parity_200.npz")]:
    print(f"{'ok ' if p.exists() else '!! '} {name:20} {p}")
    assert p.exists(), p
print(RESULT)

## `dreams_infer` (embedded verbatim, checked against the source the Kaggle leg verified)

In [ ]:
DREAMS_INFER_SRC = '"""DreaMS embedding inference, vendored: torch + numpy only (specs_novel plan §1.3 A2, roadmap P0.1).\n\nThe upstream package (pluskal-lab/DreaMS @ dbec3a0, local `DreamS/`) cannot be imported on Kaggle:\nits pins have no Python 3.12 wheels, `dreams.api` pulls in wandb / pyopenms / umap / igraph, and its\ncheckpoints pickle `dreams.*` classes. So the network code needed for the forward pass is copied here\nwith the SAME module and attribute names, which makes the upstream `state_dict` keys load `strict`.\n\n    DreamS/dreams/models/layers/fourier_features.py   FourierFeatures            (verbatim)\n    DreamS/dreams/models/layers/feed_forward.py       FeedForward                (verbatim; interpolate_interval inlined)\n    DreamS/dreams/models/dreams/layers.py             MultiheadAttention, TFeedForward (upstream\n                                                      name FeedForward), ScaleNorm, TransformerEncoder\n    DreamS/dreams/models/dreams/dreams.py             DreaMSBackbone: DreaMS.__init__ / forward /\n                                                      __normalize_spec, inference branches only\n    DreamS/dreams/models/heads/heads.py               ContrastiveHead: head(backbone(spec)[:, 0])\n\nThe weights come from `dreams_embed.pt`, written once on Colab by the P0.1a conversion (the real\npackage loads `embedding_model.ckpt` + `ssl_model.ckpt`; `args_to_dict` + `state_dict` are saved).\nFile layout: {"args_json": str, "state_dict": {name: tensor}, "meta_json": str}; plain types only, so\n`torch.load(weights_only=True)` works.\n\nPreprocessing is split in two, exactly as the cached corpus was built:\n    dreams_peaks     casmi_dreams_colab.ipynb cell 19 `_dreams_peaks` (= cell 7 `peak_list`): +2 Da cut,\n                     0.2 % floor, top-100, m/z ascending, intensity / max   -> (2, n) float32\n    to_model_input   upstream SpectrumPreprocessor(DataFormatA, n_highest_peaks=100) as used by\n                     `dreams_embeddings`: trim, zero-pad to 100, relative intensity, prepend\n                     [precursor_mz, 1.1], float32                           -> (101, 2) float32\nm/z above 1000 is NOT clipped (upstream never sets normalize_mzs); it is only outside pretraining.\n"""\nimport hashlib\nimport json\nfrom math import ceil\nfrom types import SimpleNamespace\nfrom typing import Sequence\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom torch.nn import Parameter\n\nN_PEAKS = 100             # checkpoint\'s n_highest_peaks (dreams/api.py `from_name`)\nPREC_INTENS = 1.1         # SpectrumPreprocessor default\nINT_FLOOR = 0.002         # casmi_dreams_colab.ipynb CFG\nDROP_ABOVE_PRECURSOR = 2.0\nMAX_MZ = 1000.0           # dreams/utils/dformats.py DataFormatA.max_mz\nMAX_TBXIC_STDEV = 1e-4    # DataFormatA.max_tbxic_stdev\nTRAINING_ONLY_PREFIXES = ("backbone.ff_out", "backbone.ff_out_intens", "backbone.mz_masking_loss",\n                          "backbone.ro_out")\n\n# Backbone args the inference path reads (dreams.py __init__, layers.py). Anything else is ignored.\nARG_KEYS = ("n_layers", "n_heads", "d_fourier", "d_peak", "d_mz_token", "charge_feature",\n            "fourier_strategy", "fourier_num_freqs", "fourier_min_freq", "fourier_trainable",\n            "dropout", "ff_fourier_depth", "ff_fourier_d", "no_ffs_bias", "ff_peak_depth",\n            "vanilla_transformer", "graphormer_mz_diffs", "graphormer_parametrized", "att_dropout",\n            "no_transformer_bias", "attn_mech", "ff_dropout", "residual_dropout", "pre_norm", "scnorm",\n            "dformat_max_mz", "dformat_max_tbxic_stdev")\n\n\ndef sha1(path, chunk=1 << 20):\n    h = hashlib.sha1()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(chunk), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\n# --------------------------------------------------------------------------------------------- #\n# Preprocessing\n# --------------------------------------------------------------------------------------------- #\n\ndef dreams_peaks(mz, it, prec, floor=INT_FLOOR, n=N_PEAKS, above=DROP_ABOVE_PRECURSOR):\n    mz = np.asarray(mz, np.float64); it = np.asarray(it, np.float64)\n    if len(mz) == 0: return None\n    k = mz <= prec + above; mz, it = mz[k], it[k]\n    if len(mz) == 0: return None\n    mx = it.max()\n    if not np.isfinite(mx) or mx <= 0: return None\n    k = it >= floor*mx; mz, it = mz[k], it[k]\n    if len(mz) < 2: return None\n    # Deterministic where cell 7 was not: its np.argsort (unstable) let the numpy build / CPU decide which\n    # tied peaks survive the top-n cut (G0.1-R). Ties at the cut go to the higher m/z; order is by m/z,\n    # then intensity, so the output does not depend on the input order either.\n    if len(mz) > n:\n        keep = np.lexsort((mz, it))[-n:]; mz, it = mz[keep], it[keep]\n    o = np.lexsort((it, mz)); mz, it = mz[o], it[o]\n    return np.stack([mz, it/it.max()]).astype(np.float32)   # (2, n)\n\n\ndef to_model_input(peaks, prec_mz, n=N_PEAKS, prec_intens=PREC_INTENS):\n    """(2, k) peak list -> (n + 1, 2) float32: upstream SpectrumPreprocessor.__call__ with its defaults."""\n    pl = np.array(peaks, copy=True)                                  # (2, k)\n    keep = np.sort(np.lexsort((pl[0], pl[1]))[-n:])                  # su.trim_peak_list, ties -> higher m/z\n    pl = pl[:, keep]\n    if pl.shape[1] < n:                                              # su.pad_peak_list\n        pl = np.pad(pl, ((0, 0), (0, n - pl.shape[1])), mode="constant", constant_values=0)\n    pl = np.array([pl[0], pl[1] / pl[1].max()])                      # su.to_rel_intensity\n    spec = np.vstack([np.array([prec_mz, prec_intens]), pl.T])       # su.prepend_precursor_peak\n    return spec.astype(np.float32, copy=False)\n\n\n# --------------------------------------------------------------------------------------------- #\n# Network (upstream code; see the module docstring for the file of origin)\n# --------------------------------------------------------------------------------------------- #\n\ndef interpolate_interval(a, b, n, only_inter=False, rounded=False):\n    x_min, x_max = min(a, b), max(a, b)\n    res = [x_min + i * (x_max - x_min) / (n + 1) for i in range(1 if only_inter else 0, n + 1 if only_inter else n + 2)]\n    if x_max == a:\n        res.reverse()\n    if rounded:\n        res = [round(x) for x in res]\n    return res\n\n\nclass FourierFeatures(nn.Module):\n    def __init__(self, strategy, x_min, x_max, trainable=True, funcs=\'both\', sigma=10, num_freqs=512):\n\n        assert strategy in {\'random\', \'voronov_et_al\', \'lin_float_int\'}\n        assert funcs in {\'both\', \'sin\', \'cos\'}\n        assert x_min < 1\n\n        super().__init__()\n        self.funcs = funcs\n        self.strategy = strategy\n        self.trainable = trainable\n        self.num_freqs = num_freqs\n\n        if strategy == \'random\':\n            self.b = torch.randn(num_freqs) * sigma\n        if self.strategy == \'voronov_et_al\':\n            self.b = torch.tensor(\n                [1 / (x_min * (x_max / x_min) ** (2 * i / (num_freqs - 2))) for i in range(1, num_freqs)],\n            )\n        elif self.strategy == \'lin_float_int\':\n            self.b = torch.tensor(\n                [1 / (x_min * i) for i in range(2, ceil(1 / x_min), 2)] +\n                [1 / (1 * i) for i in range(2, ceil(x_max), 1)],\n            )\n        self.b = self.b.unsqueeze(0)\n\n        self.b = nn.Parameter(self.b, requires_grad=self.trainable)\n        self.register_parameter(\'Fourier frequencies\', self.b)\n\n    def forward(self, x):\n        x = 2 * torch.pi * x @ self.b\n        if self.funcs == \'both\':\n            x = torch.cat((torch.cos(x), torch.sin(x)), dim=-1)\n        elif self.funcs == \'cos\':\n            x = torch.cos(x)\n        elif self.funcs == \'sin\':\n            x = torch.sin(x)\n        return x\n\n    def num_features(self):\n        return self.b.shape[1] if self.funcs != \'both\' else 2 * self.b.shape[1]\n\n\nclass FeedForward(nn.Module):\n\n    def __init__(self, in_dim, out_dim, hidden_dim, depth=None, act_last=True, act=nn.ReLU, bias=True, dropout=0):\n        super().__init__()\n\n        if isinstance(hidden_dim, int):\n            assert depth is not None\n            hidden_dim = [hidden_dim] * depth\n        elif hidden_dim == \'interpolated\':\n            assert depth is not None\n            hidden_dim = interpolate_interval(a=in_dim, b=out_dim, n=depth - 1, only_inter=True, rounded=True)\n        elif isinstance(hidden_dim, Sequence):  # e.g. is List or Tuple\n            depth = len(hidden_dim)\n        else:\n            raise ValueError\n\n        self.ff = nn.ModuleList([])\n        for l in range(depth):\n            d1 = hidden_dim[l - 1] if l != 0 else in_dim\n            d2 = hidden_dim[l] if l != depth - 1 else out_dim\n            self.ff.append(nn.Linear(d1, d2, bias=bias))\n            if l != depth - 1:\n                self.ff.append(nn.Dropout(p=dropout))\n            if l != depth - 1 or act_last:\n                self.ff.append(act())\n        self.ff = nn.Sequential(*self.ff)\n\n    def forward(self, x):\n        return self.ff(x)\n\n\nclass MultiheadAttention(nn.Module):\n    def __init__(self, args):\n        super(MultiheadAttention, self).__init__()\n        self.d_model = args.d_model\n        self.n_heads = args.n_heads\n        self.dropout = args.att_dropout\n        self.use_transformer_bias = not args.no_transformer_bias\n        self.attn_mech = args.attn_mech\n        self.d_graphormer_params = args.d_graphormer_params\n\n        if self.d_model % self.n_heads != 0:\n            raise ValueError(\'Required: d_model % n_heads == 0.\')\n\n        self.head_dim = self.d_model // self.n_heads\n        self.scale = self.head_dim ** -0.5\n\n        self.weights = Parameter(torch.Tensor(4 * self.d_model, self.d_model))\n        if self.use_transformer_bias:\n            self.biases = Parameter(torch.Tensor(4 * self.d_model))\n\n        if self.d_graphormer_params:\n            self.lin_graphormer = nn.Linear(self.d_graphormer_params, self.n_heads, bias=False)\n\n        mean = 0\n        std = (2 / (5 * self.d_model)) ** 0.5\n        nn.init.normal_(self.weights, mean=mean, std=std)\n        if self.use_transformer_bias:\n            nn.init.constant_(self.biases, 0.)\n\n        if self.attn_mech == \'additive_v\':\n            self.additive_v = Parameter(torch.Tensor(self.n_heads, self.head_dim))\n            nn.init.normal_(self.additive_v, mean=mean, std=std)\n\n    def forward(self, q, k, v, mask, graphormer_dists=None, do_proj_qkv=True):\n        bs, n, d = q.size()\n\n        def _split_heads(tensor):\n            bsz, length, d_model = tensor.size()\n            return tensor.reshape(bsz, length, self.n_heads, self.head_dim).transpose(1, 2)\n\n        if do_proj_qkv:\n            q, k, v = self.proj_qkv(q, k, v)\n\n        q = _split_heads(q)\n        k = _split_heads(k)\n        v = _split_heads(v)\n\n        if self.attn_mech == \'dot-product\':\n            att_weights = torch.einsum(\'bhnd,bhdm->bhnm\', q, k.transpose(-2, -1))\n        elif self.attn_mech == \'additive_v\' or self.attn_mech == \'additive_fixed\':\n            att_weights = (q.unsqueeze(-2) - k.unsqueeze(-3))\n            if self.attn_mech == \'additive_v\':\n                att_weights = (att_weights * self.additive_v.unsqueeze(0).unsqueeze(2).unsqueeze(3))\n            att_weights = att_weights.sum(dim=-1)\n        else:\n            raise NotImplementedError(f\'"{self.attn_mech}" attention mechanism is not implemented.\')\n        att_weights = att_weights * self.scale\n\n        if graphormer_dists is not None:\n            if self.d_graphormer_params:\n                att_bias = self.lin_graphormer(graphormer_dists).permute(0, 3, 1, 2)\n            else:\n                att_bias = graphormer_dists.sum(dim=-1).unsqueeze(1)\n            att_weights = att_weights + att_bias\n\n        if mask is not None:\n            att_weights.masked_fill_(mask.unsqueeze(1).unsqueeze(-1), -1e9)\n\n        att_weights = F.softmax(att_weights, dim=-1)\n        att_weights = F.dropout(att_weights, p=self.dropout, training=self.training)\n        _att_weights = att_weights.reshape(-1, n, n)\n        output = torch.bmm(_att_weights, v.reshape(bs * self.n_heads, -1, self.head_dim))\n        output = output.reshape(bs, self.n_heads, n, self.head_dim).transpose(1, 2).reshape(bs, n, -1)\n        output = self.proj_o(output)\n\n        return output, att_weights\n\n    def proj_qkv(self, q, k, v):\n        qkv_same = q.data_ptr() == k.data_ptr() == v.data_ptr()\n        kv_same = k.data_ptr() == v.data_ptr()\n\n        if qkv_same:\n            q, k, v = self._proj(q, end=3 * self.d_model).chunk(3, dim=-1)\n        elif kv_same:\n            q = self._proj(q, end=self.d_model)\n            k, v = self._proj(k, start=self.d_model, end=3 * self.d_model).chunk(2, dim=-1)\n        else:\n            q = self.proj_q(q)\n            k = self.proj_k(k)\n            v = self.proj_v(v)\n\n        return q, k, v\n\n    def _proj(self, x, start=0, end=None):\n        weight = self.weights[start:end, :]\n        bias = None if not self.use_transformer_bias else self.biases[start:end]\n        return F.linear(x, weight=weight, bias=bias)\n\n    def proj_q(self, q):\n        return self._proj(q, end=self.d_model)\n\n    def proj_k(self, k):\n        return self._proj(k, start=self.d_model, end=2 * self.d_model)\n\n    def proj_v(self, v):\n        return self._proj(v, start=2 * self.d_model, end=3 * self.d_model)\n\n    def proj_o(self, x):\n        return self._proj(x, start=3 * self.d_model)\n\n\nclass TFeedForward(nn.Module):\n    """Upstream `dreams.models.dreams.layers.FeedForward` (renamed: clashes with the peak FeedForward)."""\n    def __init__(self, args):\n        super(TFeedForward, self).__init__()\n        self.dropout = args.ff_dropout\n        self.d_model = args.d_model\n        self.ff_dim = 4 * args.d_model\n        self.use_transformer_bias = not args.no_transformer_bias\n\n        self.in_proj = nn.Linear(self.d_model, self.ff_dim, bias=self.use_transformer_bias)\n        self.out_proj = nn.Linear(self.ff_dim, self.d_model, bias=self.use_transformer_bias)\n\n        mean = 0\n        std = (2 / (self.ff_dim + self.d_model)) ** 0.5\n        nn.init.normal_(self.in_proj.weight, mean=mean, std=std)\n        nn.init.normal_(self.out_proj.weight, mean=mean, std=std)\n        if self.use_transformer_bias:\n            nn.init.constant_(self.in_proj.bias, 0.)\n            nn.init.constant_(self.out_proj.bias, 0.)\n\n    def forward(self, x):\n        y = F.relu(self.in_proj(x))\n        y = F.dropout(y, p=self.dropout, training=self.training)\n        return self.out_proj(y)\n\n\nclass ScaleNorm(nn.Module):\n    def __init__(self, scale, eps=1e-5):\n        super(ScaleNorm, self).__init__()\n        self.scale = Parameter(torch.tensor(scale))\n        self.eps = eps\n\n    def forward(self, x):\n        norm = self.scale / torch.norm(x, dim=-1, keepdim=True).clamp(min=self.eps)\n        return x * norm\n\n\nclass TransformerEncoder(nn.Module):\n    def __init__(self, args):\n        super(TransformerEncoder, self).__init__()\n        self.residual_dropout = args.residual_dropout\n        self.n_layers = args.n_layers\n        self.pre_norm = args.pre_norm\n\n        self.atts = nn.ModuleList([MultiheadAttention(args) for _ in range(self.n_layers)])\n        self.ffs = nn.ModuleList([TFeedForward(args) for _ in range(self.n_layers)])\n\n        num_scales = self.n_layers * 2 + 1 if self.pre_norm else self.n_layers * 2\n        if args.scnorm:\n            self.scales = nn.ModuleList([ScaleNorm(args.d_model ** 0.5) for _ in range(num_scales)])\n        else:\n            self.scales = nn.ModuleList([nn.LayerNorm(args.d_model) for _ in range(num_scales)])\n\n    def _layer_forward(self, i, x, src_mask, graphormer_dists):\n        pre_norm = self.pre_norm\n        post_norm = not pre_norm\n        att = self.atts[i]\n        ff = self.ffs[i]\n        att_scale = self.scales[2 * i]\n        ff_scale = self.scales[2 * i + 1]\n\n        residual = x\n        x = att_scale(x) if pre_norm else x\n        x, _ = att(q=x, k=x, v=x, mask=src_mask, graphormer_dists=graphormer_dists)\n        x = residual + F.dropout(x, p=self.residual_dropout, training=self.training)\n        x = att_scale(x) if post_norm else x\n\n        residual = x\n        x = ff_scale(x) if pre_norm else x\n        x = ff(x)\n        x = residual + F.dropout(x, p=self.residual_dropout, training=self.training)\n        x = ff_scale(x) if post_norm else x\n        return x\n\n    def forward(self, src_inputs, src_mask, graphormer_dists=None):\n        x = F.dropout(src_inputs, p=self.residual_dropout, training=self.training)\n        for i in range(self.n_layers):\n            x = self._layer_forward(i, x, src_mask, graphormer_dists)\n        x = self.scales[-1](x) if self.pre_norm else x\n        return x\n\n\nclass DreaMSBackbone(nn.Module):\n    """Upstream `DreaMS` (LightningModule) reduced to the modules the embedding forward pass uses."""\n\n    def __init__(self, args):\n        super().__init__()\n        if args.charge_feature:\n            raise NotImplementedError("charge_feature=True: the embedding checkpoint is used without charge")\n        if args.d_mz_token or not args.d_fourier:\n            raise NotImplementedError("only the Fourier-feature m/z encoding is vendored")\n        if args.vanilla_transformer:\n            raise NotImplementedError("only the DreaMS TransformerEncoder is vendored")\n        self.n_layers = args.n_layers\n        self.d_fourier = args.d_fourier\n        self.d_peak = args.d_peak\n        self.d_model = sum(d for d in [args.d_fourier, args.d_peak, args.d_mz_token] if d)\n        args.d_model = self.d_model\n        self.max_mz = args.dformat_max_mz\n        self.graphormer_mz_diffs = args.graphormer_mz_diffs\n        if args.graphormer_mz_diffs and args.graphormer_parametrized:\n            args.d_graphormer_params = args.d_fourier if args.d_fourier else 1\n        else:\n            args.d_graphormer_params = 0\n\n        token_dim = 2\n        self.fourier_enc = FourierFeatures(strategy=args.fourier_strategy, num_freqs=args.fourier_num_freqs,\n                                           x_min=args.dformat_max_tbxic_stdev if not args.fourier_min_freq else args.fourier_min_freq,\n                                           x_max=args.dformat_max_mz,\n                                           trainable=args.fourier_trainable)\n        self.ff_fourier = FeedForward(in_dim=self.fourier_enc.num_features(), out_dim=args.d_fourier, dropout=args.dropout,\n                                      depth=args.ff_fourier_depth, hidden_dim=args.ff_fourier_d, bias=not args.no_ffs_bias)\n        self.ff_peak = FeedForward(in_dim=token_dim, hidden_dim=args.d_peak, out_dim=args.d_peak, depth=args.ff_peak_depth,\n                                   dropout=args.dropout, bias=not args.no_ffs_bias)\n        self.transformer_encoder = TransformerEncoder(args)\n\n    def forward(self, spec):\n        padding_mask = spec[:, :, 0] == 0\n        peak_embs = self.ff_peak(spec / spec.new_tensor([self.max_mz, 1.]))\n        fourier_features = self.ff_fourier(self.fourier_enc(spec[..., [0]]))\n        spec = torch.cat([peak_embs, fourier_features], dim=-1)\n        graphormer_dists = None\n        if self.graphormer_mz_diffs:\n            graphormer_dists = fourier_features.unsqueeze(2) - fourier_features.unsqueeze(1)\n        return self.transformer_encoder(spec, padding_mask, graphormer_dists)\n\n\nclass DreamsEmbedder(nn.Module):\n    """Upstream ContrastiveHead at inference: Linear(d_model, d_model) on the precursor token."""\n\n    def __init__(self, args):\n        super().__init__()\n        self.backbone = DreaMSBackbone(args)\n        self.head = nn.Linear(self.backbone.d_model, self.backbone.d_model, bias=True)\n\n    def forward(self, spec):\n        return self.head(self.backbone(spec)[:, 0, ...])\n\n\n# --------------------------------------------------------------------------------------------- #\n# Conversion (Colab, with the real package loaded) and loading (anywhere)\n# --------------------------------------------------------------------------------------------- #\n\ndef args_to_dict(args):\n    """Upstream backbone `args` (argparse Namespace; `dformat` is a DataFormatA object) -> plain dict."""\n    src = vars(args) if not isinstance(args, dict) else args\n    out = {k: src[k] for k in ARG_KEYS if k in src}\n    dformat = src.get("dformat")\n    out["dformat_max_mz"] = float(getattr(dformat, "max_mz", MAX_MZ))\n    out["dformat_max_tbxic_stdev"] = float(getattr(dformat, "max_tbxic_stdev", MAX_TBXIC_STDEV))\n    missing = [k for k in ARG_KEYS if k not in out]\n    assert not missing, f"backbone args lack {missing}"\n    return out\n\n\ndef inference_state_dict(state_dict):\n    """Drop the training-only backbone heads that PreTrainedModel deletes after loading (api.py)."""\n    return {k: v for k, v in state_dict.items() if not k.startswith(TRAINING_ONLY_PREFIXES)}\n\n\ndef save_dreams(path, args, state_dict, meta):\n    torch.save({"args_json": json.dumps(args_to_dict(args), sort_keys=True),\n                "state_dict": {k: v.detach().cpu().float().contiguous() for k, v in\n                               inference_state_dict(state_dict).items()},\n                "meta_json": json.dumps(meta, sort_keys=True)}, path)\n\n\ndef build_embedder(args_dict):\n    return DreamsEmbedder(SimpleNamespace(**args_dict))\n\n\ndef load_dreams(path, device="cpu", expect_sha1=None):\n    """-> (DreamsEmbedder in eval / float32 on device, meta dict). strict load: every key must match."""\n    if expect_sha1 is not None:\n        got = sha1(path)\n        assert got == expect_sha1, f"{path}: sha1 {got} != expected {expect_sha1}"\n    blob = torch.load(path, map_location="cpu", weights_only=True)\n    model = build_embedder(json.loads(blob["args_json"]))\n    model.load_state_dict(blob["state_dict"], strict=True)\n    return model.to(device=device, dtype=torch.float32).eval(), json.loads(blob["meta_json"])\n\n\n@torch.inference_mode()\ndef dreams_embed(model, peak_lists, precursors, batch_size=64):\n    """peak_lists: (2, k) arrays from `dreams_peaks`; precursors: m/z. -> (n, d_model) float32 numpy."""\n    dev = next(model.parameters()).device\n    out = []\n    for lo in range(0, len(peak_lists), batch_size):\n        x = np.stack([to_model_input(p, m) for p, m in\n                      zip(peak_lists[lo:lo + batch_size], precursors[lo:lo + batch_size])])\n        out.append(model(torch.from_numpy(x).to(dev)).float().cpu().numpy())\n    d = model.head.out_features\n    return np.concatenate(out) if out else np.zeros((0, d), np.float32)\n'

got = hashlib.sha1(DREAMS_INFER_SRC.encode("utf-8")).hexdigest()
assert got == MAN["src_sha1"]["dreams_infer.py"], f"dreams_infer.py {got} != the G0 run's {MAN['src_sha1']['dreams_infer.py']}"
(SRC / "dreams_infer.py").write_text(DREAMS_INFER_SRC, encoding="utf-8")
sys.path.insert(0, str(SRC))
import dreams_infer as di
MODEL, MMETA = di.load_dreams(GATE / "dreams_embed.pt", device=DEV, expect_sha1=MAN["dreams_embed_sha1"])
RESULT["src_sha1"] = got
print(f"loaded dreams_embed.pt on {DEV}: {MMETA}")

## Row order and resume state

`corpus_ce.parquet` carries each corpus row's `train.parquet` index from the P0.2 replay. Before
trusting it, the identity columns at those `train.parquet` rows are compared with
`corpus_meta.parquet` once more.

In [ ]:
t0 = time.time()
META = pd.read_parquet(DOUT / "corpus_meta.parquet", columns=["ingest_lib", "inchikey14", "adduct",
                                                                "precursor_mz", "oob_mass"])
N = len(META)
TROW = pq.read_table(GATE / "corpus_ce.parquet", columns=["row", "train_row"]).to_pandas()
assert len(TROW) == N and (TROW.row.to_numpy() == np.arange(N)).all()
TROW = TROW.train_row.to_numpy(np.int64)
ids = pq.read_table(TRAIN, columns=["inchikey14", "adduct", "precursor_mz"]).to_pandas()
N_TRAIN = len(ids)
ids = ids.iloc[TROW].reset_index(drop=True)
order_bad = {c: int((ids[c].astype(str).to_numpy() != META[c].astype(str).to_numpy()).sum())
             for c in ("inchikey14", "adduct")}
order_bad["precursor_mz"] = int((ids.precursor_mz.to_numpy(np.float64) != META.precursor_mz.to_numpy(np.float64)).sum())
assert not any(order_bad.values()), f"corpus_ce train_row does not line up with corpus_meta: {order_bad}"
PREC = META.precursor_mz.to_numpy(np.float64)
del ids

def chunk_bounds(n, chunk):
    return [(lo, min(lo + chunk, n)) for lo in range(0, n, chunk)]

def chunk_done(path, rows):
    if not path.exists():
        return False
    try:
        a = np.load(path, mmap_mode="r")
    except Exception:
        return False
    return a.shape == (rows, DIM) and a.dtype == np.float32

BOUNDS = chunk_bounds(N, CHUNK)
TODO = [i for i, (lo, hi) in enumerate(BOUNDS) if not chunk_done(OUT / f"emb_{i:04d}.npy", hi - lo)]
RESULT.update(rows=N, chunks=len(BOUNDS), order_mismatches=order_bad)
save_result()
print(f"{N:,} rows in {len(BOUNDS)} chunks; to embed: {TODO} ({time.time() - t0:.0f}s)")

## Peaks for every corpus row

One streaming pass over `train.parquet`, `dreams_peaks` per corpus row (= cell 7 `peak_list`,
tested offline). All rows are prepared even when resuming, because the peak counts feed the
new-vs-old breakdown and the parity rows are checked against `parity_200.npz`.

In [ ]:
t0 = time.time()
pos = np.full(N_TRAIN, -1, np.int64)
pos[TROW] = np.arange(N)
PLS = [None] * N

def list_parts(col):
    return col.offsets.to_numpy(), col.values.to_numpy(zero_copy_only=False)

base = 0
for batch in pq.ParquetFile(TRAIN).iter_batches(batch_size=100_000,
                                                 columns=["ms2_mzs", "ms2_normalized_intensities"]):
    n = batch.num_rows
    hit = np.flatnonzero(pos[base:base + n] >= 0)
    if len(hit):
        mo, mv = list_parts(batch.column(0))
        io_, iv = list_parts(batch.column(1))
        for local in hit:
            j = pos[base + local]
            PLS[j] = di.dreams_peaks(mv[mo[local]:mo[local + 1]], iv[io_[local]:io_[local + 1]], PREC[j])
    base += n
assert base == N_TRAIN
lost = [j for j, p in enumerate(PLS) if p is None]
assert not lost, f"{len(lost)} corpus rows have no usable peaks (first: {lost[:5]}) -- the replay said they had"
NPK = np.array([p.shape[1] for p in PLS], np.int16)
RESULT["t_peaks_s"] = round(time.time() - t0)
save_result()
print(f"peaks for {N:,} rows ({time.time() - t0:.0f}s); peaks kept: median {np.median(NPK):.0f}, max {NPK.max()}")

## Embed, chunk by chunk

Each chunk is written locally, copied to Drive under a temporary name and renamed, so a chunk on
Drive is always complete.

In [ ]:
di.dreams_embed(MODEL, PLS[:8], PREC[:8])                  # warm-up
t_all = time.time()
for i in TODO:
    lo, hi = BOUNDS[i]
    t0 = time.time()
    E = di.dreams_embed(MODEL, PLS[lo:hi], PREC[lo:hi], batch_size=BATCH)
    torch.cuda.synchronize(DEV)
    assert E.shape == (hi - lo, DIM) and E.dtype == np.float32 and np.isfinite(E).all(), (i, E.shape)
    local = WORK / f"emb_{i:04d}.npy"
    np.save(local, E)
    tmp = OUT / f"emb_{i:04d}.npy.tmp"
    shutil.copyfile(local, tmp)
    os.replace(tmp, OUT / f"emb_{i:04d}.npy")
    local.unlink()
    dt = time.time() - t0
    print(f"chunk {i:04d} rows {lo:,}-{hi:,}: {dt:.0f}s ({1000 * dt / (hi - lo):.2f} ms/spectrum)", flush=True)
RESULT["t_embed_s"] = round(time.time() - t_all)
RESULT["embedded_chunks_this_session"] = TODO
save_result()

## G0.1-R — the new cache on the parity rows

Pre-registered: the 200 parity rows' peaks rebuilt here equal `parity_200.npz`, and the new cache
vectors have cosine ≥ 0.999 with Colab (b) on every row. The old-cache numbers are printed for
contrast.

In [ ]:
NEW = [np.load(OUT / f"emb_{i:04d}.npy", mmap_mode="r") for i in range(len(BOUNDS))]
OLD = [np.load(DOUT / "emb" / f"emb_{i:04d}.npy", mmap_mode="r") for i in range(len(BOUNDS))]
assert sum(a.shape[0] for a in NEW) == N and all(a.shape[1] == DIM and a.dtype == np.float32 for a in NEW)

def rows_of(chunks, rows):
    return np.stack([np.asarray(chunks[r // CHUNK][r % CHUNK]) for r in rows]).astype(np.float32)

def rowcos(x, y):
    x = x.astype(np.float64); y = y.astype(np.float64)
    return (x * y).sum(1) / (np.linalg.norm(x, axis=1) * np.linalg.norm(y, axis=1))

Z = np.load(GATE / "parity_200.npz")
prow, poff = Z["row"], Z["off"]
peaks_same = all(np.array_equal(PLS[r][0], Z["mz"][poff[k]:poff[k + 1]]) and
                 np.array_equal(PLS[r][1], Z["it"][poff[k]:poff[k + 1]]) and PREC[r] == Z["prec"][k]
                 for k, r in enumerate(prow))
cos_new = rowcos(rows_of(NEW, prow), Z["emb_b"])
cos_old = rowcos(rows_of(OLD, prow), Z["emb_b"])
finite = all(bool(np.isfinite(np.asarray(a)).all()) for a in NEW)
G01R = bool(peaks_same and finite and cos_new.min() >= COS_CACHE)
RESULT["G0.1-R"] = dict(rows=N, finite=finite, parity_peaks_identical=peaks_same,
                        cos_new_vs_b_min=float(cos_new.min()), cos_new_vs_b_median=float(np.median(cos_new)),
                        cos_old_vs_b_min=float(cos_old.min()), threshold=COS_CACHE, passed=G01R)
save_result()
print(f"parity peaks identical: {peaks_same}; all finite: {finite}")
print(f"new cache vs Colab (b): min {cos_new.min():.9f}  median {np.median(cos_new):.9f}")
print(f"old cache vs Colab (b): min {cos_old.min():.6f}")
print(f"G0.1-R: {'PASS' if G01R else 'FAIL'}")

## New vs old cache over the whole corpus (recorded, not gating)

How far the old `_dreams_out/emb/` vectors are from the pinned model, row by row: overall, and by
library, adduct, precursor above 1000 Da and peak-count bin. Anything that still reads the old
cache can use this to judge whether it matters.

In [ ]:
t0 = time.time()
COS = np.empty(N, np.float64)
for i, (lo, hi) in enumerate(BOUNDS):
    for a in range(0, hi - lo, 50_000):
        b = min(a + 50_000, hi - lo)
        COS[lo + a:lo + b] = rowcos(np.asarray(NEW[i][a:b]), np.asarray(OLD[i][a:b]))
np.save(OUT / "cos_new_vs_old.npy", COS.astype(np.float32))

QS = [0.0, 0.001, 0.01, 0.1, 0.5]

def summarise(cos, groups):
    """Per group: n, min, the low quantiles and the share below 0.999, worst groups first."""
    df = pd.DataFrame({"g": groups, "cos": cos})
    out = df.groupby("g").cos.agg(n="size", min="min", median="median",
                                  below_0999=lambda s: float((s < 0.999).mean()))
    return out.sort_values("min").reset_index()

overall = {f"q{q:g}": float(np.quantile(COS, q)) for q in QS}
overall.update(below_0999=float((COS < 0.999).mean()), below_0995=float((COS < 0.995).mean()),
               below_099=float((COS < 0.99).mean()))
BY = {"ingest_lib": META.ingest_lib.astype(str).to_numpy(), "adduct": META.adduct.astype(str).to_numpy(),
      "oob_mass": META.oob_mass.to_numpy(), "npk_bin": np.digitize(NPK, [10, 40])}
by = {k: summarise(COS, v).to_dict("records") for k, v in BY.items()}
RESULT["new_vs_old"] = dict(overall=overall, by=by, seconds=round(time.time() - t0))
save_result()
print(json.dumps(overall, indent=1))
for k in BY:
    print(f"\nby {k}:\n{summarise(COS, BY[k]).head(12).to_string(index=False)}")

## Manifest and the G0 verdict

In [ ]:
MANIFEST = dict(
    rows=N, chunks=len(BOUNDS), chunk_rows=CHUNK, dim=DIM, dtype="float32",
    order="row i = corpus_meta.parquet row i (P0.2 replay; train.parquet index in corpus_ce.parquet train_row)",
    dreams_commit=DREAMS_COMMIT, dreams_embed_sha1=MAN["dreams_embed_sha1"], dreams_infer_sha1=RESULT["src_sha1"],
    gpu=RESULT["gpu"], torch=RESULT["torch"], batch_size=BATCH, tf32=False,
    chunk_sha1={f"emb_{i:04d}.npy": di.sha1(OUT / f"emb_{i:04d}.npy") for i in range(len(BOUNDS))},
    g01r=RESULT["G0.1-R"], built=time.strftime("%Y-%m-%d %H:%M"))
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=1))

legs = dict(official=bool(P0["P0.1a"]["leg_official"]), kaggle=KAGGLE_COS_MIN >= COS_CACHE, cache=G01R)
others = {k: v for k, v in P0["items"].items() if k != "G0.1"}
G01 = all(legs.values())
HEAD = "PASS" if G01 and all(v.startswith("PASS") for v in others.values()) else "HOLD"
RESULT.update(g01_legs=legs, G01="PASS" if G01 else "HOLD", verdict=HEAD, finished=time.strftime("%Y-%m-%d %H:%M"))
save_result()

print("=" * 88)
print(f"G0 VERDICT: {HEAD}")
print("=" * 88)
print(f"G0.1  {'PASS' if G01 else 'HOLD'}  legs {legs}")
for k, v in others.items():
    print(f"{k}  {v}")
print(f"\nG0.1-R  new vs (b): min {cos_new.min():.9f} (>= {COS_CACHE}); parity peaks identical {peaks_same}; "
      f"rows {N:,}; chunks {len(BOUNDS)}")
print(f"new vs old: min {overall['q0']:.6f}, p0.1% {overall['q0.001']:.6f}, median {overall['q0.5']:.6f}; "
      f"share < 0.999: {overall['below_0999']:.4f}")
print(f"\nnew cache: {OUT}  (manifest.json, cos_new_vs_old.npy)")
print(f"full report: {GATE / 'reembed_gate.json'}")